In [1]:
import sys
from pathlib import Path

# Add the parent directory to sys.path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import gymnax 
import jax.numpy as jnp
import jax
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from core.helpers import initialize_evaluator, make_env
from core.utils import load_run_data
import marimo as mo
from core.mail import email_pdf
from core import networks

📧 Email functions ready!
Try: test_email() first, then email_pdf_simple()


### 1. Random Policy on Four Rooms.

In [2]:
config, td_metrics = load_run_data('random/td_exact/tuned_saved_metrics/', 'FourRooms-misc', '../results')

from core.bellman_error import value_metrics_light


env, env_params = make_env(config)
evaluator = initialize_evaluator(config, env, env_params)
obs_shape = env.observation_space(env_params).shape
n_actions = env.action_space(env_params).n
S = evaluator.obs_stack
rng = jax.random.PRNGKey(config.get('SEED', 42))

# random network... needed as input to value metrics light.
network, network_params = networks.initialize_network(
    rng, obs_shape, env, env_params, 16, n_heads=1, layer_norm=config['LAYER_NORM']
)
train_state = networks.initialize_flax_train_state(config, network, network_params)

diagnostics_fr_random = value_metrics_light(evaluator, network, network_params, random_policy = True)
diagnostics_fr_random['non_reversible_coeff']


Env: FourRooms-misc
Default Obs Shape: (13, 13, 2)
Obs Shape: (13, 13, 2)
Action Shape: ()
- obs shape for initialization is  (13, 13, 2)
number of features is  16


Array(8.631478e-08, dtype=float32)

In [3]:
from core.utils import load_run_data_from_path
config, trained_metrics = load_run_data_from_path('../results/ppo/ground_truth/four_rooms_250/FourRooms-misc')


In [4]:
_, out = load_run_data_from_path('../results/ppo/ground_truth/four_rooms_250/FourRooms-misc') 
policy_train_state = out['runner_state'][0]
policy_params = jax.tree_util.tree_map(lambda x: x[0], policy_train_state.params)
get_policy = lambda obs: policy_train_state.apply_fn(policy_params, obs)[0]

def get_policy_matrix():
    "produces pi(.|S) where S is all states"
    pi_dist, _ = policy_train_state.apply_fn(policy_params, evaluator.obs_stack)
    pi = pi_dist.probs
    terminal_policy = jnp.ones( [1,n_actions], dtype=pi.dtype) / n_actions
    pi = jnp.vstack([pi, terminal_policy])
    return pi

    Pi = get_policy_matrix()

from core.bellman_error import value_metrics_light

env, env_params = make_env(config)
evaluator = initialize_evaluator(config, env, env_params)
obs_shape = env.observation_space(env_params).shape
n_actions = env.action_space(env_params).n
S = evaluator.obs_stack
rng = jax.random.PRNGKey(config.get('SEED', 42))

# random network... needed as input to value metrics light.
network, network_params = networks.initialize_network(
    rng, obs_shape, env, env_params, 16, n_heads=1, layer_norm=config['LAYER_NORM']
)
train_state = networks.initialize_flax_train_state(config, network, network_params)

diagnostics_fr_policy = value_metrics_light(evaluator, network, network_params, target_policy_fn = get_policy)
diagnostics_fr_policy['non_reversible_coeff'] # ||DP^T - P^T D||^2 / S


Env: FourRooms-misc
Default Obs Shape: (13, 13, 2)
Obs Shape: (13, 13, 2)
Action Shape: ()
- obs shape for initialization is  (13, 13, 2)
number of features is  16


Array(0.00025367, dtype=float32)

In [9]:
config, trained_metrics = load_run_data_from_path('../results/ppo/ground_truth/mountain_car_250/MountainCar-v0')
_, out = load_run_data_from_path('../results/ppo/ground_truth/mountain_car_250/MountainCar-v0') 
policy_train_state = out['runner_state'][0]
policy_params = jax.tree_util.tree_map(lambda x: x[0], policy_train_state.params)
get_policy = lambda obs: policy_train_state.apply_fn(policy_params, obs)[0]

def get_policy_matrix():
    "produces pi(.|S) where S is all states"
    pi_dist, _ = policy_train_state.apply_fn(policy_params, evaluator.obs_stack)
    pi = pi_dist.probs
    terminal_policy = jnp.ones( [1,n_actions], dtype=pi.dtype) / n_actions
    pi = jnp.vstack([pi, terminal_policy])
    return pi

    Pi = get_policy_matrix()

from core.bellman_error import value_metrics_light

env, env_params = make_env(config)
evaluator = initialize_evaluator(config, env, env_params)
obs_shape = env.observation_space(env_params).shape
n_actions = env.action_space(env_params).n
S = evaluator.obs_stack
rng = jax.random.PRNGKey(config.get('SEED', 42))

# random network... needed as input to value metrics light.
network, network_params = networks.initialize_network(
    rng, obs_shape, env, env_params, 16, n_heads=1, layer_norm=config['LAYER_NORM']
)
train_state = networks.initialize_flax_train_state(config, network, network_params)

diagnostics_mc_policy = value_metrics_light(evaluator, network, network_params, target_policy_fn = get_policy)
diagnostics_mc_policy['non_reversible_coeff']


Env: MountainCar-v0
Default Obs Shape: (2,)
Obs Shape: (2,)
Action Shape: ()
- obs shape for initialization is  (2,)
number of features is  16


Array(6.0982455e-07, dtype=float32)